In [ ]:
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 67.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import textstat
import re, json, nltk, os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from wordcloud import WordCloud
from tkinter.constants import E

In [ ]:
# Finding text colulumns

RAW_DIR = Path("/content/drive/MyDrive/data_LIWC")

FILES = [
    "blacklivesmatter.csv",
    "ukraine_war1.csv",
    "ukraine_war2.csv",
    "ariana_grande_attack.csv",
    "charlie_hebdo.csv",
    "berlin_attack.csv",
]

potential_text = ["text", "F", "L", "V", "body", "description", "name"]
liwc_features = ["Tone", "affect", "posemo", "negemo", "anx", "anger", "sad"]

def preview(s:str) -> str:
  s = re.sub(r"\s+", " ", str(s)).strip()
  return s[:100]

def textlike(series: pd.Series) -> float:
  s = series.dropna().astype(str).head(200)
  if len(s) == 0:
    return 0.0
  # share of rows with spaces + letters
  has_space = s.str.contains(r"\s").mean()
  has_letters = s.str.contains(r"[a-zA-Z]").mean()
  avg_len = s.str.len().mean()
  looks_numeric = s.str.match(r"^\d+(\.\d+)?$").mean()
  return (has_space * 0.4 + has_letters * 0.4 + min(avg_len/100,1.0) *0.2) * (1 - looks_numeric)

def inspect (fname, nrows = 2000, samples = 3):
  path = RAW_DIR / fname
  print("="*50)
  print(f"File:{fname}")
  print("="*50)

  df = pd.read_csv(path, nrows=nrows, dtype = str)
  print(f"\nRows loaded:{len(df):,} | Cols: {len(df.columns)}")

  present_features = [col for col in liwc_features if col in df.columns]
  print(f"\nLIWC features present: {present_features}")

  priority = [col for col in potential_text if col in df.columns]

  # fallback based on score
  obj_cols = [col for col in df.columns if df[col].dtype == "object" and col not in priority]
  scored = sorted([(textlike(df[col]), col) for col in obj_cols], reverse = True)[:5]
  second_best = [col for score, col in scored if score >0.25]

  candidates= []
  for col in priority + second_best:
    if col not in candidates:
      candidates.append(col)

  print("\nCandidates for text column:", candidates[:10])

  for col in candidates[:5]:
    s = df[col].dropna().astype(str).head(200)
    s = s[s.str.strip().str.len()>0]
    print(f"\nSamples from '{col}': (score = {textlike(df[col]):.2f})")
    for text in s.head(samples).tolist():
      print(f" - {preview(text)}")

  if present_features:
    print("\nLIWC features:")
    print(df[present_features].head(5).to_string(index= False))

  print("\n")

for fname in FILES:
  inspect(fname)

File:blacklivesmatter.csv

Rows loaded:2,000 | Cols: 184

LIWC features present: ['Tone', 'affect', 'posemo', 'negemo', 'anx', 'anger', 'sad']

Candidates for text column: ['F', 'L', 'V', 'body', 'BX', 'AX', 'BJ', 'AU', 'AI']

Samples from 'F': (score = 1.00)
 - Acting DHS Chief Says Feds Are �Working On� Arresting #BlackLivesMatter <U+270A><U+0001F3FD><U+270A>
 - Donald Trump says #BlackLivesMatter <U+270A><U+0001F3FD><U+270A><U+0001F3FE><U+270A><U+0001F3FF> #BL
 - �The first time I ever heard of #BlackLivesMatter <U+270A><U+0001F3FD><U+270A><U+0001F3FE><U+270A><U

Samples from 'L': (score = 0.41)
 - FALSE
 - FALSE
 - FALSE

Samples from 'V': (score = 0.55)
 - https://talkingpointsmemo.com/news/acting-dhs-chief-says-feds-are-working-on-arresting-blm-leaders
 - https://thegrio.com/2020/09/01/trump-blm-bad/
 - https://thegrio.com/2020/09/01/trump-blm-bad/

Samples from 'body': (score = 0.01)
 - 0,00
 - 0,00
 - 0,00

Samples from 'BX': (score = 1.00)
 - Human.. I'm for: #fairness, #happi

In [ ]:
# Data Sampling Main

CONFIG = {
    "input_dir" : "/content/drive/MyDrive/data_LIWC/",
    "output_dir" : "/content/drive/MyDrive/submission_data/",
    "sample_size_per_dataset" : 10000,
    "test_split_size" : 0.2,
    "random_state" : 42,

    "text_col_map":{
        "blacklivesmatter": "F",
        "ukraine_war1": "text",
        "ukraine_war2": "text",
        "ariana_grande_attack": "V",
        "charlie_hebdo": "L",
        "berlin_attack": "L",
    },

    "domains":{
        "social_movement": ["blacklivesmatter"],
        "crisis": ["ukraine_war1", "ukraine_war2"],
        "terror_attack": ["ariana_grande_attack", "berlin_attack", "charlie_hebdo"]

    },

    "emotion_columns": ["anx", "anger", "sad"]
}

def normalize_text(s: str) -> str:
    s = str(s)

    # LIWC placeholders / html leftovers
    s = re.sub(r"<U\+[0-9A-Fa-f]+>", " ", s)
    s = s.replace("&amp;", "&")

    # urls/mentions/hashtags
    s = re.sub(r"http\S+|www\S+|https\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    s = re.sub(r"#\w+", " ", s)

    # whitespace + case
    s = s.replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

class DataLoader:
  @staticmethod
  def load_data(file_path: str, dataset_name: str) -> pd.DataFrame:
    print(f"Loading {dataset_name}...")
    try:
      df = pd.read_csv(file_path, low_memory= False)
      print(f"Loaded {dataset_name} with {len(df):,} rows")
      return df
    except Exception as e:
      print(f"Error loading {dataset_name}: {e}")
      return pd.DataFrame()

class Standardizer:
  @staticmethod
  def standardize_data(df: pd.DataFrame, dataset_name: str, config: dict) -> pd.DataFrame:
    print(f"Standardizing {dataset_name}...")

    df_std = df.copy()
    text_column = None

    # text column mapping
    if dataset_name in config["text_col_map"]:
      mapped_col = config["text_col_map"][dataset_name]
      print(f"All columns: {list(df_std.columns)}")

      if mapped_col in df_std.columns:
        text_column = mapped_col
        print(f"Mapped {dataset_name} to {text_column}")
      else:
        print(f"Warning: {dataset_name} has no column named {mapped_col}")
    '''
    if text_column is None:
      text_column = "text"
      print(f"Using default text column: {text_column}")
    else:
      print("No text column found. Skipping.")
    '''
    # cleaning NA's
    df_std["text"] = df_std[text_column].astype(str)
    df_std["text"] = df_std["text"].replace(
        {
              "nan": "",
              "NaN": "",
              "None": "",
              "null": "",
              "c(NA, NA)": "",
              "c(NA, NA, NA, NA, NA, NA, NA, NA)": "",
        }
    )

    # too short text
    before = len(df_std)
    df_std = df_std[df_std["text"].str.len()>10].copy()
    removed = before - len(df_std)
    if removed >0:
      print(f"Removed {removed} rows with empty/short text")

    # dedup before sampling
    df_std["join_key"] = df_std["text"].apply(normalize_text)
    before = len(df_std)
    df_std = df_std[df_std["join_key"].str.len()>0].copy()
    df_std = df_std.drop_duplicates(subset=["join_key"]).copy()
    print(f"Removed {before-len(df_std)} rows with duplicate text from {dataset_name}")

    # unique smaple
    k = config.get("sample_size_per_dataset", None)
    if k and len(df_std) > k:
      df_std = df_std.sample(n=k, random_state=config.get("random_state", 42)).copy()
      print(f"Sampled unique {dataset_name} to {len(df_std):,}")

    # attaching labels
    for emotion_col in config['emotion_columns']:
      if emotion_col in df_std.columns:
        label_col = f"label{emotion_col}"

        def convert_liwc_score(x):
          try:
            if isinstance(x, str):
              x = x.replace(",",".")
            score = float(x)
            return 1 if score > 0.1 else 0
          except:
            return 0

        df_std[label_col] = df_std[emotion_col].apply(convert_liwc_score)
        print(f"{emotion_col} converted to {label_col}: {int(df_std[label_col].sum()):,} samples")

    # metadata
    df_std["dataset_name"] = dataset_name
    df_std["domain"] = Standardizer._get_domain(dataset_name, config)

    avg_text_length = df_std["text"].apply(len).mean() if len(df_std) else 0
    print(f"Average text length: {avg_text_length:.1f} chars")
    if len(df_std) > 0:
      print(f"Sample text: '{df_std['text'].iloc[0][:100]}...'")

    # output columns
    keep_cols = ["text", "dataset_name", "domain", "join_key"]
    for emotion_col in config['emotion_columns']:
      label_col = f"label{emotion_col}"
      if label_col in df_std.columns:
        keep_cols.append(label_col)

    return df_std[keep_cols]
  @staticmethod
  def _get_domain(dataset_name: str, config: dict) -> str:
    dataset_lower = dataset_name.lower()
    for domain, datasets in config["domains"].items():
      for keyword in datasets:
        if keyword in dataset_lower:
          return domain
    return "unknown"

def prepare_data(config = CONFIG):
  print("Preparing data...")
  print("="*50)

  os.makedirs(config["output_dir"], exist_ok=True)
  input_dir = Path(config["input_dir"])
  csv_files = list(input_dir.glob("*.csv"))

  if not csv_files:
    print(f"No CSV files found in {input_dir}")
    return

  print(f"Found {len(csv_files)} CSV files:")
  all_datasets ={}

  # process each one
  for file_path in tqdm(csv_files):
    dataset_name = file_path.stem

    if dataset_name not in config["text_col_map"]:
      print(f"Warning: {dataset_name} not in mapping. Skipping.")
      continue

    print(f"\n{'='*50}")
    print(f"  Processing: {dataset_name}...")
    print("="*50)

    df = DataLoader.load_data(str(file_path), dataset_name)
    if df.empty:
      print(f"Failed to load {dataset_name}")
      continue

    df_std = Standardizer.standardize_data(df, dataset_name, config)
    if df_std.empty:
      print(f"Failed to standardize {dataset_name}")
      continue

    all_datasets[dataset_name] = df_std
    print(f"Prepared {len(df_std):,} unique samples")

    dataset_path = Path(config["output_dir"]) / f"{dataset_name}_prepared.csv"
    df_std.drop(columns=["join_key"]).to_csv(dataset_path, index=False)
    print(f"Saved prepared data to {dataset_path}")

  if not all_datasets:
    print("No datasets to combine")
    return

  print(f"\n{'='*50}")
  print("\nCombining all datasets...")
  print("="*50)

  combined_df = pd.concat(all_datasets.values(), ignore_index=True)

  # global dedup + stable row_id
  before = len(combined_df)
  combined_d = combined_df.drop_duplicates(subset=["join_key"]).copy()
  combined_df = combined_df.reset_index(drop=True )
  print(f"Removed {before-len(combined_df)} rows with duplicate text")

  combined_df["row_id"] = np.arange(len(combined_df), dtype=int)

  combined_path = Path(config["output_dir"]) / "all_data_combined.csv"
  combined_df.drop(columns=["join_key"]).to_csv(combined_path, index=False)
  print(f"Saved combined data to {combined_path}")

  #cross-domain split
  print(f"\n{'='*50}")
  print("\nCreating cross-domain splits")
  print("="*50)

  train_domains = ["social_movement", "crisis"]
  test_domains = ["terror_attack"]

  train_val_df = combined_df[combined_df["domain"].isin(train_domains)].copy()
  test_df = combined_df[combined_df["domain"].isin(test_domains)].copy()

  print(f"Train/Val samples: {len(train_val_df):,} samples")
  print(f"Test samples: {len(test_df):,} samples")

  if len(train_val_df) > 0:
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=config["test_split_size"],
        random_state=config["random_state"],
        stratify=train_val_df["domain"]
    )
  else:
    train_df = pd.DataFrame()
    val_df = pd.DataFrame()

  out_dir = Path(config["output_dir"])
  train_path = out_dir / "train_data.csv"
  val_path = out_dir / "val_data.csv"
  test_path = out_dir / "test_data.csv"

  train_df.drop(columns=["join_key"]).to_csv(train_path, index=False)
  val_df.drop(columns=["join_key"]).to_csv(val_path, index=False)
  test_df.drop(columns=["join_key"]).to_csv(test_path, index=False)

  print(f"Train: {train_path} ({len(train_df):,} samples)")
  print(f"Validation: {val_path} ({len(val_df):,} samples)")
  print(f"Test: {test_path} ({len(test_df):,} samples)")

  print("\nDone!")

  stats = {
      "total_samples_unique": int(len(combined_df)),
      "processed_datasets": list(all_datasets.keys()),
      "domain_distribution": combined_df["domain"].value_counts().to_dict(),
      "split_sizes": {"train": int(len(train_df)), "validation" : int(len(val_df)), "test" : int(len(test_df))},
      "text_lenght_stats": {
          "mean": float(combined_df["text"].str.len().mean()),
          "min": int(combined_df["text"].str.len().min()),
          "max": int(combined_df["text"].str.len().max()),
          "std": float(combined_df["text"].str.len().std()),
      },
  }

  stats_path = out_dir / "stats.json"
  with open(stats_path, "w") as f:
    json.dump(stats, f, indent=2)

  print(f"Stats saved to {stats_path}")
  print("\nDomain distribution:")
  for domain, count in stats["domain_distribution"].items():
    print(f" - {domain}: {count:,} ")

  print("Preparation complete.")
  print(f"Output directory:{config['output_dir']}")

  return{
      "train_df": train_df.drop(columns=["join_key"]),
      "val_df": val_df.drop(columns=["join_key"]),
      "test_df": test_df.drop(columns=["join_key"]),
      "combined_df": combined_df.drop(columns=["join_key"]),
      "stats": stats
  }

if __name__ == "__main__":
  prepare_data()

Preparing data...
Found 6 CSV files:


  0%|          | 0/6 [00:00<?, ?it/s]


  Processing: blacklivesmatter...
Loading blacklivesmatter...
Loaded blacklivesmatter with 847,872 rows
Standardizing blacklivesmatter...
All columns: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'AA', 'AB', 'AC', 'AD', 'AE', 'AF', 'AG', 'AH', 'AI', 'AJ', 'AK', 'AL', 'AM', 'AN', 'AO', 'AP', 'AQ', 'AR', 'AS', 'AT', 'AU', 'AV', 'AW', 'AX', 'AY', 'AZ', 'BA', 'BB', 'BC', 'BD', 'BE', 'BF', 'BG', 'BH', 'BI', 'BJ', 'BK', 'BL', 'BM', 'BN', 'BO', 'BP', 'BQ', 'BR', 'BS', 'BT', 'BU', 'BV', 'BW', 'BX', 'BY', 'BZ', 'CA', 'CB', 'CC', 'CD', 'CE', 'CF', 'CG', 'CH', 'CI', 'CJ', 'CK', 'CL', 'CM', 'WC', 'Analytic', 'Clout', 'Authentic', 'Tone', 'WPS', 'Sixltr', 'Dic', 'function', 'pronoun', 'ppron', 'i', 'we', 'you', 'shehe', 'they', 'ipron', 'article', 'prep', 'auxverb', 'adverb', 'conj', 'negate', 'verb', 'adj', 'compare', 'interrog', 'number', 'quant', 'affect', 'posemo', 'negemo', 'anx', 'anger', 'sad', 'social', '

 17%|█▋        | 1/6 [01:27<07:15, 87.19s/it]

Prepared 10,000 unique samples
Saved prepared data to /content/drive/MyDrive/submission_data/blacklivesmatter_prepared.csv

  Processing: ukraine_war1...
Loading ukraine_war1...
Loaded ukraine_war1 with 2,282,233 rows
Standardizing ukraine_war1...
All columns: ['Unnamed: 0', 'Unnamed: 0.1', 'author_id', 'created_at', 'geo', 'tweet_id', 'lang', 'retweet_count', 'reply_count', 'quote_count', 'like_count', 'source', 'text', 'referenced_type', 'referenced_id', 'conversation_id', 'followers_count', 'following_count', 'tweet_count', 'account_created_at', 'username', 'verified', 'description', 'name', 'location_user\r', 'Analytic', 'Clout', 'Authentic', 'Tone', 'affect', 'posemo', 'negemo', 'anx', 'anger', 'sad']
Mapped ukraine_war1 to text
Removed 49 rows with empty/short text


...

 83%|████████▎ | 5/6 [06:29<01:15, 75.60s/it]

Saved prepared data to /content/drive/MyDrive/submission_data/charlie_hebdo_prepared.csv

  Processing: berlin_attack...
Loading berlin_attack...
Loaded berlin_attack with 108,271 rows
Standardizing berlin_attack...
All columns: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'WC', 'Analytic', 'Clout', 'Authentic', 'Tone', 'WPS', 'Sixltr', 'Dic', 'function', 'pronoun', 'ppron', 'i', 'we', 'you', 'shehe', 'they', 'ipron', 'article', 'prep', 'auxverb', 'adverb', 'conj', 'negate', 'verb', 'adj', 'compare', 'interrog', 'number', 'quant', 'affect', 'posemo', 'negemo', 'anx', 'anger', 'sad', 'social', 'family', 'friend', 'female', 'male', 'cogproc', 'insight', 'cause', 'discrep', 'tentat', 'certain', 'differ', 'percept', 'see', 'hear', 'feel', 'bio', 'body', 'health', 'sexual', 'ingest', 'drives', 'affiliation', 'achieve', 'power', 'reward', 'risk', 'focuspast', 'focuspresent', 'focusfuture', 'relativ', 'motion', 'space

100%|██████████| 6/6 [06:35<00:00, 65.87s/it]

Removed 72766 rows with duplicate text from berlin_attack
Sampled unique berlin_attack to 10,000
anx converted to labelanx: 1,634 samples
anger converted to labelanger: 2,488 samples
sad converted to labelsad: 1,279 samples
Average text length: 111.2 chars
Sample text: 'Horrible, horrible! This is becoming really common occurrence. #BerlinAttack...'
Prepared 10,000 unique samples
Saved prepared data to /content/drive/MyDrive/submission_data/berlin_attack_prepared.csv


Combining all datasets...
Removed 0 rows with duplicate text


Saved combined data to /content/drive/MyDrive/submission_data/all_data_combined.csv


Creating cross-domain splits
Train/Val samples: 30,000 samples
Test samples: 30,000 samples
Train: /content/drive/MyDrive/submission_data/train_data.csv (24,000 samples)
Validation: /content/drive/MyDrive/submission_data/val_data.csv (6,000 samples)
Test: /content/drive/MyDrive/submission_data/test_data.csv (30,000 samples)

Done!
Stats saved to /content/drive/MyDrive/submission_data/stats.json

Domain distribution:
 - terror_attack: 30,000 
 - crisis: 20,000 
 - social_movement: 10,000 
Preparation complete.
Output directory:/content/drive/MyDrive/submission_data/


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import textstat
import re, json, nltk, os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from wordcloud import WordCloud
from tkinter.constants import E

# Adding emotional features drom LIWC
# run after initial sampling

CONFIG = {
    "raw_dir": "/content/drive/MyDrive/data_LIWC",
    "prep_dir": "/content/drive/MyDrive/submission_data",

    "text_column_mapping": {
        "blacklivesmatter": "F",
        "ukraine_war1": "text",
        "ukraine_war2": "text",
        "ariana_grande_attack": "V",
        "charlie_hebdo": "L",
        "berlin_attack": "L",
    },

    "exclude_always": {
        "text", "raw_text", "join_key","dataset_name", "domain", "row_id",
        "id", "docid", "doc_id", "document_id", "uid","user_id",
        "timestamp", "time", "date",
        "source", "url", "link"
    },

    "liwc_detection": {
        "sample_size": 200,
        "min_numeric_values": 3,
        "min_text_length": 10,
    },

    "output_files": {
        "per_dataset_suffix": "_with_liwc_features.csv",
        "combined_file": "all_data_combined.csv",
        "split_files": {
            "train": "train_data.csv",
            "val": "val_data.csv",
            "test": "test_data.csv"
        },
        "column_list": "liwc_all_columns.json"
    },

    # Split names to process
    "splits": ["train_data", "val_data", "test_data"]
}

def normalize_text(s: str) -> str:
    s = str(s)

    # LIWC placeholders / html leftovers
    s = re.sub(r"<U\+[0-9A-Fa-f]+>", " ", s)
    s = s.replace("&amp;", "&")

    # urls/mentions/hashtags
    s = re.sub(r"http\S+|www\S+|https\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    s = re.sub(r"#\w+", " ", s)

    # whitespace + case
    s = s.replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

def to_float_mixed(t):
    if pd.isna(t):
      return None
    t = str(t).strip()
    if t == "" or t.lower() in {"nan", "none", "null"}:
      return None
    t = t.replace(",", ".")
    try:
      return float(t)
    except:
      return None

def find_features(raw_df: pd.DataFrame, raw_text_col, config: dict) -> list:
  exclude = config["exclude_always"]
  detection = config["liwc_detection"]

  candidates = [
      col for col in raw_df.columns
      if col != raw_text_col and col not in exclude
  ]

  features_cols = []
  for col in candidates:
    series = raw_df[col]
    sample = series.dropna().astype(str).head(detection["sample_size"])
    if sample.empty:
      continue

    converted = sample.apply(to_float_mixed)
    if converted.notna().sum() >= detection["min_numeric_values"]:
      features_cols.append(col)

  return features_cols

def load_raw_data(ds: str, raw_text_col: str, config: dict):
    raw_path = Path(config["raw_dir"]) / f"{ds}.csv"
    detection = config["liwc_detection"]

    if not raw_path.exists():
      print(f"File not found: {raw_path}")
      return None

    raw = pd.read_csv(raw_path, dtype=str)

    if raw_text_col not in raw.columns:
      print(f"Column not found: {raw_text_col} in {ds}")
      return None

    features = find_features(raw, raw_text_col, config)

    if not features:
      print(f"No features found in {ds}")
      features = []

    keep = [raw_text_col] + features
    raw = raw[keep].copy().rename(columns={raw_text_col: "raw_text"})

    # join key
    raw["join_key"] = raw["raw_text"].astype(str).apply(normalize_text)
    raw = raw[raw["join_key"].str.len()> detection["min_text_length"]].copy()
    raw = raw.drop_duplicates(subset=["join_key"]).copy()

    for col in features:
      raw[col] = raw[col].apply(to_float_mixed)

    return raw[["join_key"] + features]

def load_prep_data(ds: str, config: dict):
    prep_path = Path(config["prep_dir"]) / f"{ds}_prepared.csv"
    detection = config["liwc_detection"]

    if not prep_path.exists():
      print(f"File not found: {prep_path}")
      return None

    prep = pd.read_csv(prep_path, dtype=str)
    prep["join_key"] = prep["text"].astype(str).apply(normalize_text)
    prep = prep[prep["join_key"].str.len()> detection["min_text_length"]].copy()
    prep = prep.drop_duplicates(subset=["join_key"]).copy()

    return prep

def attach_liwc(ds:str, config: dict):
    raw_text_col = config["text_column_mapping"][ds]
    out_path = Path(config["prep_dir"]) / f"{ds}{config['output_files']['per_dataset_suffix']}"

    prep = load_prep_data(ds, config)
    if prep is None:
      print(f"Skipping {ds}: missing prepared file")
      return None

    raw_processed = load_raw_data(ds, raw_text_col, config)
    if raw_processed is None:
      print(f"Skipping {ds}: missing raw file")
      return None

    # liwc_cols excluding join_key
    liwc_cols = [col for col in raw_processed.columns if col != "join_key"]

    out = prep.merge(raw_processed, on="join_key", how="left")

    # match rate
    if liwc_cols:
      match_rate = out[liwc_cols].notna().any(axis=1).mean()
      print(f"{ds}: prep={len(prep):,} | raw={len(raw_processed):,} | "
            f"features={len(liwc_cols)} | match={match_rate*100:.1f}%")
    else:
      print(f"{ds}: prep={len(prep):,} | raw={len(raw_processed):,} | features = 0")

    out.drop(columns=["join_key"]).to_csv(out_path, index=False)
    print(f"Saved to {out_path}")

    return liwc_cols

def raw_liwc_lookup(datasets: list, feature_union : list, config: dict) -> pd.DataFrame:
    frames = []

    for ds in datasets:
      raw_text_col = config["text_column_mapping"][ds]
      raw_processed = load_raw_data(ds, raw_text_col, config)

      if raw_processed is None:
        print(f"Skipping {ds}: missing raw file")
        continue

      cols_present = [col for col in feature_union if col in raw_processed.columns]
      frames.append(raw_processed[["join_key"] + cols_present])

    if not frames:
      return pd.DataFrame(columns =["join_key" + str(list(feature_union))])

    raw_all = pd.concat(frames, ignore_index=True)
    cols = [ col for col in raw_all.columns if col != "join_key"]

    if cols:
      raw_all["_nn"] = raw_all[cols].notna().sum(axis=1)

      raw_all = raw_all.sort_values(["join_key", "_nn"], ascending=[True, False])
      raw_all = raw_all.drop_duplicates(subset=["join_key"], keep="first")
      raw_all = raw_all.drop(columns=["_nn"])

    return raw_all

def attach_liwc_to_combined(feature_union: list,datasets: list, config: dict):
    prep_dir = Path(config["prep_dir"])
    out_files = config["output_files"]

    combined_path = prep_dir / "all_data_combined.csv"
    if not combined_path.exists():
      print(f"File not found: {combined_path}")
      return

    combined = pd.read_csv(combined_path, low_memory=False)
    combined["join_key"] = combined["text"].astype(str).apply(normalize_text)

    #global lookup
    raw_lookup = raw_liwc_lookup(datasets, feature_union, config)
    liwc_cols = [col for col in feature_union if col in raw_lookup.columns]
    combined_out = prep_dir / out_files["combined_file"]
    combined_with_features = combined.merge(
          raw_lookup[["join_key"] + liwc_cols],
          on="join_key",
          how="left"
    )

    if liwc_cols:
        match_rate = combined_with_features[liwc_cols].notna().any(axis=1).mean()
        print(f"[combined] total={len(combined):,} | features={len(liwc_cols)} | "
                f"match={match_rate*100:.1f}%")

    combined_with_features.drop(columns=["join_key"]).to_csv(combined_out, index=False)
    print(f"  saved: {combined_out}")

    #
    combined_with_features["join_key"] = combined_with_features["text"].astype(str).apply(normalize_text)

    for split in config["splits"]:
        split_in = prep_dir / f"{split}.csv"
        split_out = prep_dir / out_files["split_files"][split.split('_')[0]]

        if not split_in.exists():
            print(f"  Skipping {split}: file not found")
            continue

        split_df = pd.read_csv(split_in, low_memory=False)
        split_df["join_key"] = split_df["text"].astype(str).apply(normalize_text)

        merged = split_df.merge(
            combined_with_features[["join_key"] + liwc_cols],
            on="join_key",
            how="left",
        ).drop(columns=["join_key"])

        merged.to_csv(split_out, index=False)
        print(f"  saved: {split_out} ({len(merged):,} rows)")

def run_attachment(config: dict = None):
      if config is None:
          config = CONFIG

      config["raw_dir"] = Path(config["raw_dir"])
      config["prep_dir"] = Path(config["prep_dir"])
      config["prep_dir"].mkdir(parents=True, exist_ok=True)

      print("=" * 50)
      print("Attaching features.")
      print("=" * 50)
      print(f"Raw data: {config['raw_dir']}")
      print(f"Prepared data: {config['prep_dir']}")
      print("=" * 50)

      datasets = list(config["text_column_mapping"].keys())

      print("\n Processing per dataset")
      feature_union = set()
      per_ds_features = {}

      for ds in datasets:
          print(f"\n--- {ds} ---")
          cols = attach_liwc(ds, config)
          if cols is None:
              continue
          per_ds_features[ds] = cols
          feature_union.update(cols)

      feature_union = sorted(feature_union)
      print(f"\nTotal unique LIWC features detected: {len(feature_union)}")

      cols_path = config["prep_dir"] / config["output_files"]["column_list"]
      with open(cols_path, "w") as f:
          json.dump({
              "liwc_all_columns": feature_union,
              "per_dataset_columns": {k: list(v) for k, v in per_ds_features.items()}
          }, f, indent=2)
      print(f"Saved LIWC column list: {cols_path}")

      print("\n" + "=" * 50)
      print("Processing splits.")
      print("=" * 50)

      attach_liwc_to_combined(feature_union, datasets, config)

      print("\n" + "=" * 50)
      print("Attachment complete.")
      print("=" * 50)

if __name__ == "__main__":
      run_attachment()

Attaching features.
Raw data: /content/drive/MyDrive/data_LIWC
Prepared data: /content/drive/MyDrive/submission_data

 Processing per dataset

--- blacklivesmatter ---
blacklivesmatter: prep=9,857 | raw=174,078 | features=120 | match=100.0%
Saved to /content/drive/MyDrive/submission_data/blacklivesmatter_with_liwc_features.csv

--- ukraine_war1 ---
ukraine_war1: prep=9,937 | raw=172,949 | features=23 | match=100.0%
Saved to /content/drive/MyDrive/submission_data/ukraine_war1_with_liwc_features.csv

--- ukraine_war2 ---
ukraine_war2: prep=9,943 | raw=164,908 | features=23 | match=100.0%
Saved to /content/drive/MyDrive/submission_data/ukraine_war2_with_liwc_features.csv

--- ariana_grande_attack ---
ariana_grande_attack: prep=9,910 | raw=526,645 | features=104 | match=100.0%
Saved to /content/drive/MyDrive/submission_data/ariana_grande_attack_with_liwc_features.csv

--- charlie_hebdo ---
charlie_hebdo: prep=9,887 | raw=210,335 | features=104 | match=100.0%
Saved to /content/drive/MyDrive

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
plt.rc('font', size=14)
print("Data Analytics")

data_path = Path("/content/drive/MyDrive/submission_data/")
liwc_path = Path("/content/drive/MyDrive/data_LIWC/")

prepared_files = list(data_path.glob("*.csv"))
print(f"Found {len(prepared_files)} prepared datasets:")
for f in sorted(prepared_files):
    print(f" - {f.stem.replace('_prepared', '')}")

liwc_files = list(liwc_path.glob("*.csv"))
print(f"\nFound {len(liwc_files)} LIWC datasets:")
for f in sorted(liwc_files):
    print(f" - {f.stem}")

dataset_metrics = []
for prep_file in prepared_files:
  dataset_name = prep_file.stem.replace('_prepared', '')
  liwc_file = liwc_path / f"{dataset_name}.csv"

  print(f"\n{'='*50}")
  print(f"Dataset: {dataset_name}")
  print(f"{'='*50}")

  try:
    df_prep = pd.read_csv(prep_file)

    metrics = {
        'dataset': dataset_name,
        'rows': df_prep.shape[0],
        'samples': len(df_prep),
        'columns_prep': df_prep.shape[1],
        'missing_prep': df_prep.isna().sum().sum(),
    }

    text_length = df_prep['text'].astype(str).apply(len)
    word_count = df_prep['text'].astype(str).apply(lambda x: len(x.split()))

    metrics.update({
        'avg_chars': text_length.mean(),
        'avg_words': word_count.mean(),
        'min_chars': text_length.min(),
        'max_chars': text_length.max(),
        'min_words': word_count.min(),
        'max_words': word_count.max(),
        'unique_words': df_prep['text'].astype(str).apply(lambda x: len(set(x.split()))).mean(),
        'vocab_size': df_prep['text'].astype(str).apply(lambda x: len(set(x.split()))).max(),
    })

    dataset_metrics.append(metrics)

    print(f"Samples:{metrics['samples']}")
    print(f"Text columns: {list(df_prep.columns)}")

    print(f"\nText Statistics:")
    print(f" - Avg characters: {metrics['avg_chars']:.0f}")
    print(f" - Avg words: {metrics['avg_words']:.0f}")
    print(f" - Range: {metrics['min_words']} - {metrics['max_words']} words")
    print(f" - Range: {metrics['min_chars']} - {metrics['max_chars']} characters")
    print(f" - Vocabulary size: {metrics['vocab_size']:,}")
    print(f" - Missing values: {metrics['missing_prep']}")

    print(f"\nSample Texts:")
    for i,text in enumerate(df_prep['text'].head(3).values):
      preview = text[150] + "..." if len(text) > 150 else text
      print(f" - {i+1}: {preview}")

  except Exception as e:
    continue

  try:
    df_liwc = pd.read_csv(liwc_file)

    metrics = {
        'dataset': dataset_name,
        'columns_liwc': df_liwc.shape[1],
        'missing_liwc': df_liwc.isna().sum().sum(),
    }

    dataset_metrics.append(metrics)

    print(f"Samples:{metrics['samples']}")

    if df_liwc is not None:
      print(f"LIWC columns: {df_liwc.shape[1]} (first 10: {list(df_liwc.columns[:10])})")
      print(f"LIWC emotion features: {[col for col in df_liwc.columns if 'emo' in col.lower()or 'anx' in col.lower() or 'anger' in col.lower() or 'sad' in col.lower()]}")

  except Exception as e:
    continue

summary_df = pd.DataFrame(dataset_metrics)
print("\nSummary across datasets")
print(summary_df.to_string())

print("\nDomain Analysis")
data_combined = data_path / "all_data_combined.csv"
if data_combined.exists():
  df_combined = pd.read_csv(data_combined)
  print(f"\nTotal combined samples: {len(df_combined)}")

  if 'domain' in df_combined.columns:
    domain_counts = df_combined['domain'].value_counts()
    print("\nDomain Counts:")
    for domain, count in domain_counts.items():
      percentage = (count / len(df_combined)) * 100
      print(f" - {domain}: {count:,} samples ({percentage:.1f}%)")

  if 'dataset_name' in df_combined.columns:
    dataset_counts = df_combined['dataset_name'].value_counts()
    print("\nDataset Distribution:")
    for dataset, count in dataset_counts.head(10).items():
      percentage = (count / len(df_combined)) * 100
      print(f" - {dataset}: {count:,} samples ({percentage:.1f}%)")

print("\nText Quality Analysis")

def text_quality(text_series):
  texts = text_series.astype(str)

  metrics = {
      'total_texts': len(texts),
      'unique_texts': len(texts.unique()),
      'empty_texts': (texts == '').sum(),
      'short_texts': (texts.apply(len) < 10).sum(),
      'avg_sentence_length': texts.apply(lambda x: len(re.findall(r'\w+', x))).mean(),
      'avg_word_length': texts.apply(lambda x: np.mean([len(word) for word in x.split()]) if x.split() else 0).mean(),
      'hashtag_count': texts.apply(lambda x: x.count('#')).sum(),
      'mention_count': texts.apply(lambda x: x.count('@')).sum(),
      'url_count': texts.apply(lambda x: x.count('http')).sum(),
      'emoji_count': texts.apply(lambda x: len(re.findall(r'[^\w\s]', x))).sum(),
      'duplicate_texts': texts.duplicated().sum(),
  }

  return metrics

for prep_file in prepared_files:
  dataset_name = prep_file.stem.replace('_prepared', '')
  df_prep = pd.read_csv(prep_file)

  if 'text' in df_prep.columns:
    metrics = text_quality(df_prep['text'])
    print(f"\nText Quality Metrics for {dataset_name}:")
    for metric, value in metrics.items():
      print(f" - {metric}: {value}")

summary_df = pd.DataFrame(dataset_metrics)
summary_df.to_csv(data_path / "dataset_metrics.csv", index=False)
print(f"\nMetrics saved to {data_path / 'dataset_metrics.csv'}")

Data Analytics
Found 17 prepared datasets:
 - all_data_combined
 - ariana_grande_attack
 - ariana_grande_attack_with_liwc_features
 - berlin_attack
 - berlin_attack_with_liwc_features
 - blacklivesmatter
 - blacklivesmatter_with_liwc_features
 - charlie_hebdo
 - charlie_hebdo_with_liwc_features
 - dataset_metrics
 - test_data
 - train_data
 - ukraine_war1
 - ukraine_war1_with_liwc_features
 - ukraine_war2
 - ukraine_war2_with_liwc_features
 - val_data

Found 6 LIWC datasets:
 - ariana_grande_attack
 - berlin_attack
 - blacklivesmatter
 - charlie_hebdo
 - ukraine_war1
 - ukraine_war2

Dataset: blacklivesmatter
Samples:10000
Text columns: ['text', 'dataset_name', 'domain', 'labelanx', 'labelanger', 'labelsad']

Text Statistics:
 - Avg characters: 205
 - Avg words: 26
 - Range: 2 - 96 words
 - Range: 24 - 1224 characters
 - Vocabulary size: 91
 - Missing values: 0

Sample Texts:
 - 1: s...
 - 2: 
...
 - 3: THIS is why we march. THIS is why we will never stop. #BlackLivesMatter https://t.c